# 🧹 TIỀN XỬ LÝ, LỌC RÁC VÀ LÀM SẠCH DỮ LIỆU GỘP (DATA CLEANING)
**Môn học**: IT4930 - Khoa học dữ liệu (Data Science)  
**Thành viên thực hiện**: Quỳnh Anh (TECH-01) - **Task ID**: SP1-T3  
**Dữ liệu đầu vào**: `data/interim/merged_traffic_data.csv` (Đã tích hợp từ task SP1-T2)  
**Dữ liệu đầu ra**: `data/processed/clean_traffic_data.csv`  

---

### 🎯 Mục tiêu kỹ thuật & Tiêu chí nghiệm thu (Definition of Done - DoD):
1. **Loại bỏ các cột trùng lặp hoặc không mang giá trị dự báo**:
   - Cột thời gian thô chưa chuẩn hóa: `date_time_raw`, `date_time` (đã có trường chuẩn ISO-8601 `date_time_iso`, `date_time_local`, `date_time_utc`).
2. **Phát hiện và xử lý giá trị ngoại lai vật lý (Physical Outliers & Infeasible Values)**:
   - Nhiệt độ vô lý: Phát hiện các mốc `0.0 Kelvin` (-273.15°C - lỗi sensor) và các giá trị ngoài biên vật lý Trái Đất `[200K, 330K]`.
   - Lượng mưa vô lý: Phát hiện lượng mưa âm `< 0` hoặc các đột biến phi lý do lỗi sensor như `9831.3 mm/h` (gần 10 mét nước/giờ!).
   - Lưu lượng xe vô lý: Xử lý các điểm âm `< 0` hoặc sụt giảm về `0` bất thường trong khung giờ cao điểm trên đường cao tốc I-94.
   - Lượng tuyết và độ che phủ mây: Đảm bảo tuyết không âm và mây nằm trong đoạn `[0, 100]%`.
3. **Bù khuyết dữ liệu thiếu (Time-series Imputation)**:
   - Biến liên tục (`traffic_volume`, `temp`, `clouds_all`, `wind_speed_ms`): Áp dụng **Nội suy tuyến tính (Linear Interpolation)** kết hợp Forward-fill (`ffill`) và Backward-fill (`bfill`).
   - Biến thời tiết thưa (`rain_1h`, `snow_1h`): Bù khuyết dựa trên ngữ cảnh thời tiết thực tế (`weather_main`).
   - Biến nhãn ngày lễ (`holiday_name`): Bù `'Not Holiday'` cho các ngày thường để tránh việc Pandas tự động nhận diện `'None'` thành `NaN` khi đọc từ file CSV.
4. **Nghiệm thu toàn diện (DoD)**:
   - Dữ liệu sạch 100% không còn chứa `NaN` khi nạp bằng `pd.read_csv()`.
   - Không còn ngoại lai vật lý.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập font và style vẽ biểu đồ
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Thêm đường dẫn src để import pipeline chuẩn
sys.path.append(os.path.abspath('..'))
from src.data_cleaning import (
    load_dataset,
    drop_redundant_features,
    sanitize_physical_outliers,
    impute_missing_values,
    validate_clean_data,
    export_clean_data,
    PHYSICAL_BOUNDS
)

print("✅ Import thành công module làm sạch dữ liệu!")

## 1. Nạp và Khảo sát Toàn diện Dữ liệu Gộp (Interim Data Profiling)
Nạp file kết quả từ giai đoạn tích hợp `data/interim/merged_traffic_data.csv` để thống kê hiện trạng khuyết thiếu và ngoại lai.

In [ ]:
INTERIM_FILE = '../data/interim/merged_traffic_data.csv'
df_interim = load_dataset(INTERIM_FILE)

print(f"Kích thước dữ liệu gộp ban đầu: {df_interim.shape[0]:,} dòng, {df_interim.shape[1]} cột")
print("\nDanh sách các cột dữ liệu:")
print(list(df_interim.columns))
df_interim.head(3)

### 📊 Phân tích định lượng tỷ lệ khuyết thiếu (Missing Data Analysis)
Xem xét số lượng giá trị `NaN` trên từng cột trước khi tiến hành xử lý.

In [ ]:
# Tính toán tỷ lệ missing theo từng cột
missing_counts = df_interim.isna().sum()
missing_pcts = (missing_counts / len(df_interim)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing_counts,
    'Missing_Percentage': missing_pcts
}).query('Missing_Count > 0').sort_values(by='Missing_Count', ascending=False)

print("Bảng thống kê các thuộc tính khuyết thiếu:")
display(missing_df)

# Trực quan hóa tỷ lệ missing
plt.figure(figsize=(10, 4))
missing_features = missing_df.drop(index=['holiday_name'], errors='ignore')
sns.barplot(x=missing_features['Missing_Percentage'], y=missing_features.index, palette='Blues_r')
plt.title('Tỷ lệ khuyết thiếu (%) trên các biến quan trắc (loại trừ holiday_name)', fontsize=13, fontweight='bold')
plt.xlabel('Tỷ lệ Missing (%)')
plt.ylabel('Thuộc tính')
plt.tight_layout()
plt.show()

## 2. Loại bỏ các Cột Trùng lặp & Không mang Giá trị Dự báo (Feature Dropping)
Trong bảng gộp, có hai cột chuỗi thô `date_time_raw` và `date_time` (định dạng `M/D/YY H:MM`). Cả hai cột này hoàn toàn trùng lặp với `date_time_iso` và không còn giá trị phân tích.

In [ ]:
df_dropped, drop_stats = drop_redundant_features(df_interim)
print(f"Số lượng cột ban đầu: {drop_stats['initial_columns_count']}")
print(f"Đã loại bỏ {drop_stats['num_dropped_columns']} cột: {drop_stats['dropped_columns']}")
print(f"Số lượng cột còn lại: {drop_stats['remaining_columns_count']}")

## 3. Phát hiện và Xử lý Ngoại lai Vật lý (Physical Outlier Sanitization)
Các lỗi cảm biến nghiêm trọng trong dữ liệu khí tượng và lưu lượng:
1. **Nhiệt độ 0.0 K**: Độ không tuyệt đối (-273.15°C) là phi thực tế trong tự nhiên.
2. **Lượng mưa cực đại 9,831.3 mm**: Tương đương 9.8 mét nước chỉ trong 1 giờ - lỗi cảm biến ghi nhận giá trị rác.
3. **Lưu lượng xe bằng 0**: Xuất hiện vào chiều tối thứ 7 trên tuyến cao tốc liên bang I-94.

In [ ]:
# Khảo sát chi tiết ngoại lai nhiệt độ 0 K
temp_zeros = df_dropped[df_dropped['temp'] == 0]
print(f"Số bản ghi có nhiệt độ 0.0 K: {len(temp_zeros)}")
display(temp_zeros[['date_time_iso', 'temp', 'weather_main', 'weather_description']].head(5))

# Khảo sát ngoại lai lượng mưa lớn nhất
extreme_rain = df_dropped[df_dropped['rain_1h'] > 300]
print(f"\nSố bản ghi có lượng mưa > 300 mm/h: {len(extreme_rain)}")
display(extreme_rain[['date_time_iso', 'rain_1h', 'weather_main', 'weather_description']])

# Tiến hành cô lập và gán các ngoại lai vật lý về NaN
df_sanitized, outlier_stats = sanitize_physical_outliers(df_dropped)
print("\nThống kê các ngoại lai đã được đưa về NaN để chuẩn bị nội suy:")
for k, v in outlier_stats.items():
    print(f"- {k}: {v}")

## 4. Bù khuyết Dữ liệu theo Đặc thù Chuỗi Thời gian (Time-series Imputation)
Áp dụng kết hợp:
- **Nội suy tuyến tính (Linear Interpolation)**: Thích hợp cho các đại lượng đo biến thiên liên tục theo thời gian (`temp`, `traffic_volume`, `wind_speed_ms`).
- **Forward-fill (ffill)**: Áp dụng cho các trạng thái thời tiết phân loại (`weather_main`, `weather_description`) do tính chất duy trì ổn định của các hiện tượng khí quyển.
- **Domain Fill (0.0)**: Áp dụng cho lượng mưa/tuyết khi trời không có mưa.

In [ ]:
df_clean, impute_stats = impute_missing_values(df_sanitized)

print(f"Tổng số giá trị khuyết thiếu trước khi bù: {impute_stats['total_missing_before']:,}")
print(f"Tổng số giá trị khuyết thiếu sau khi bù: {impute_stats['total_missing_after']}")
print(f"Kích thước bộ dữ liệu sau làm sạch: {df_clean.shape}")

### 📈 Trực quan hóa kết quả làm sạch: Đoạn chuỗi thời gian trước và sau khi bù khuyết

In [ ]:
# Minh họa đoạn dữ liệu nhiệt độ từng bị dính lỗi 0 K vào ngày 2014-01-31
sample_range = (pd.to_datetime(df_clean['date_time_iso']) >= '2014-01-30') & (pd.to_datetime(df_clean['date_time_iso']) <= '2014-02-01')

plt.figure(figsize=(14, 5))
dates = pd.to_datetime(df_clean.loc[sample_range, 'date_time_iso'])
raw_temps = df_interim.loc[sample_range, 'temp']
clean_temps = df_clean.loc[sample_range, 'temp']

plt.plot(dates, clean_temps, label='Nhiệt độ sau nội suy tuyến tính (Clean)', color='#2b8a3e', linewidth=2.5)
plt.scatter(dates[raw_temps == 0], raw_temps[raw_temps == 0], color='red', s=80, zorder=5, label='Điểm lỗi 0.0 K (Raw)')
plt.title('Đối sánh Nhiệt độ trước và sau khi làm sạch (Đợt sensor lỗi ngày 31/01/2014)', fontsize=13, fontweight='bold')
plt.xlabel('Thời gian')
plt.ylabel('Nhiệt độ (Kelvin)')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Kiểm định Chất lượng Dữ liệu Chuẩn (QA Checklist & Definition of Done)
Đảm bảo các tiêu chí nghiệm thu được thỏa mãn 100% trước khi xuất file.

In [ ]:
qa_results = validate_clean_data(df_clean)
qa_summary = pd.DataFrame(list(qa_results.items()), columns=['Tiêu chí kiểm định (QA Rule)', 'Kết quả'])
qa_summary['Trạng thái'] = qa_summary['Kết quả'].apply(lambda x: 'PASSED ✅' if x else 'FAILED ❌')
display(qa_summary[['Tiêu chí kiểm định (QA Rule)', 'Trạng thái']])

# Kiểm tra xác nhận không còn bất kỳ giá trị NaN nào
assert df_clean.isna().sum().sum() == 0, "Vẫn còn tồn tại giá trị NaN!"
print("\n🎉 XÁC NHẬN: TẤT CẢ CÁC TIÊU CHÍ ĐÃ ĐẠT CHUẨN DEFINITION OF DONE!")

## 6. Xuất Dữ liệu Chuẩn Sạch (Export Cleaned Dataset)
Lưu file kết quả vào thư mục `data/processed/clean_traffic_data.csv`.

In [ ]:
PROCESSED_FILE = '../data/processed/clean_traffic_data.csv'
export_clean_data(df_clean, PROCESSED_FILE)
print("Hoàn tất bàn giao sản phẩm SP1-T3!")